# Detector de Spam en Correos

**Evaluación Grupal 1 · Lenguaje de Programación II · Semestre 2026-2**

---

### Equipo de trabajo

| N° | Integrante | Usuario |
|:--:|------------|:--:|
| 1 | Huamani Diestra, Miriam Vanessa | MVanessaHD |
| 2 | Meza Asto, Angel David | AngelMol0810 |
| 3 | Chavez Cuzcano, Victor Javier | jchest771-stack |
| 4 | Alva Aquino, Brian | blolchevere-ctrl |

**Docente:** Vargas Paredes, Ana Cecilia

---

**Contenido:** 
1. Descripción del sistema 
2. Arquitectura 
3. Implementación 
4. Ejecución de prueba 
5. Menú interactivo 
6. Informe de clases 
7. Conceptos de POO 
8. Conclusiones


## 1. Descripción del sistema

Simulación de un **filtro de spam**. Cada correo se revisa con un conjunto de *reglas*; cada regla aporta puntos de "sospecha". El detector suma los puntos y, si el total alcanza un **umbral**, marca el correo como spam y guarda los **motivos** que lo explican.

**Funcionalidades**

- Agregar correos a una bandeja.
- Analizar cada correo con varias reglas.
- Mostrar el puntaje y los motivos de cada decisión.
- Listar correos spam y normales.
- Cambiar el umbral de sensibilidad.
- Sumar una regla opcional de **Naive Bayes** básico, que aprende de ejemplos y se integra como una regla más.

## 2. Arquitectura y diseño

```
Bandeja ──contiene──▶ Correo ◀──analiza── DetectorSpam ──tiene──▶ Regla
                                                                    ▲
         ┌──────────────────┬─────────────────┬──────────────┬───┴────────────┬────────────────┐
 ReglaPalabrasClave   ReglaRemitente   ReglaEnlaces   ReglaMayusculas   ReglaNaiveBayes
```

| Relación | Tipo | Significado |
|---|---|---|
| `Bandeja` → `Correo` | Composición | La bandeja guarda una lista de correos |
| `DetectorSpam` → `Regla` | Composición | El detector guarda una lista de reglas |
| `DetectorSpam` ⇢ `Correo` | Dependencia | El detector analiza y modifica un correo |
| `ReglaXxx` ⇒ `Regla` | Herencia | Cada regla especializa a la clase base |

**Flujo de una decisión:** `Bandeja.agregar(correo)` → `DetectorSpam.analizar(correo)` → cada `Regla.evaluar(correo)` devuelve `(puntaje, motivo)` → el detector suma, guarda el puntaje en el correo y lo marca como spam o normal → `Bandeja.listar_spam()` y `Bandeja.resumen()` muestran el resultado.

**Contrato entre clases**

| Clase | Método | Recibe | Devuelve |
|---|---|---|---|
| `Regla` y sus hijas | `evaluar(correo)` | un `Correo` | tupla `(puntaje, motivo)` |
| `DetectorSpam` | `analizar(correo)` | un `Correo` | `True` si es spam |
| `Bandeja` | `listar_spam()`, `listar_normales()` | — | lista de `Correo` |

## 3. Implementación

El orden de las celdas importa: cada clase se define antes de usarse.

### 3.1 Clase `Correo`
Representa un mensaje y guarda el resultado de su análisis.

In [ ]:
class Correo:
    """Un correo con sus datos y el resultado del análisis de spam.

    Los atributos son privados (encapsulamiento). Se leen mediante propiedades
    y el puntaje se valida en su setter.
    """

    def __init__(self, remitente, asunto, cuerpo):
        self.__remitente = remitente
        self.__asunto = asunto
        self.__cuerpo = cuerpo
        self.__puntaje = 0
        self.__es_spam = False
        self.__motivos = []

    @property
    def remitente(self):
        return self.__remitente

    @property
    def asunto(self):
        return self.__asunto

    @property
    def cuerpo(self):
        return self.__cuerpo

    @property
    def puntaje(self):
        return self.__puntaje

    @puntaje.setter
    def puntaje(self, valor):
        if valor < 0:
            raise ValueError("El puntaje no puede ser negativo")
        self.__puntaje = valor

    @property
    def es_spam(self):
        return self.__es_spam

    @property
    def motivos(self):
        # Devuelve una copia para no exponer la lista interna
        return list(self.__motivos)

    def agregar_motivo(self, motivo):
        """Registra por qué una regla sospechó de este correo."""
        self.__motivos.append(motivo)

    def limpiar_motivos(self):
        """Borra los motivos de un análisis anterior."""
        self.__motivos.clear()

    def marcar_como_spam(self):
        self.__es_spam = True

    def marcar_como_normal(self):
        self.__es_spam = False

    def __str__(self):
        estado = "SPAM" if self.__es_spam else "NORMAL"
        return f"[{estado}] ({self.__puntaje} pts) {self.__remitente} - {self.__asunto}"

**Resumen de métodos de `Correo`**

| Método | Qué hace |
|---|---|
| `__init__(remitente, asunto, cuerpo)` | Crea el correo con sus datos; deja el puntaje en 0, no spam y sin motivos |
| `remitente`, `asunto`, `cuerpo` | Propiedades de solo lectura: devuelven los datos originales |
| `puntaje` | Lee el puntaje; al asignarlo, valida que no sea negativo |
| `es_spam` | Indica si el correo fue marcado como spam |
| `motivos` | Devuelve una copia de la lista de motivos (la interna no se expone) |
| `agregar_motivo(motivo)` | Registra por qué una regla sospechó del correo |
| `limpiar_motivos()` | Borra los motivos de un análisis anterior |
| `marcar_como_spam()`, `marcar_como_normal()` | Cambian el estado del correo |
| `__str__()` | Texto legible: estado, puntaje, remitente y asunto |

### 3.2 Clase base `Regla` y sus hijas (herencia)
`Regla` es una clase **abstracta**: define qué debe saber hacer cualquier regla (`evaluar`) pero no se puede instanciar. Cada regla hija **hereda** `nombre` y `peso` y la especializa implementando `evaluar(correo)`, que devuelve la tupla `(puntaje, motivo)` o `(0, "")` si la regla no se cumple. Como el detector solo llama a `evaluar`, puede usar cualquier regla sin conocer su tipo (**polimorfismo**).

In [ ]:
import math
import re
import unicodedata
from abc import ABC, abstractmethod


def normalizar(texto):
    """Pasa a minúsculas y quita tildes, para que 'PREMIÓ' y 'premio' coincidan."""
    descompuesto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in descompuesto if unicodedata.category(c) != "Mn")


class Regla(ABC):
    """Clase base abstracta: cada regla aporta puntos de sospecha a un correo.

    Es abstracta porque no tiene sentido crear una 'Regla' genérica: solo se
    usan sus hijas. Cada hija DEBE implementar ``evaluar`` (polimorfismo).
    """

    def __init__(self, nombre, peso):
        if peso <= 0:
            raise ValueError("El peso de una regla debe ser positivo")
        self.__nombre = nombre
        self.__peso = peso

    @property
    def nombre(self):
        return self.__nombre

    @property
    def peso(self):
        return self.__peso

    @abstractmethod
    def evaluar(self, correo):
        """Recibe un Correo y devuelve la tupla (puntaje, motivo).

        Si la regla no se cumple devuelve (0, "").
        """

    def __str__(self):
        return f"{self.nombre} (peso {self.peso})"


class ReglaPalabrasClave(Regla):
    """Suma ``peso`` por cada palabra sospechosa distinta (acepta plurales)."""

    PALABRAS_POR_DEFECTO = ("gratis", "premio", "urgente", "ganaste", "oferta")

    def __init__(self, palabras=None, peso=2):
        super().__init__("Palabras clave", peso)
        self.__palabras = tuple(palabras or self.PALABRAS_POR_DEFECTO)

    def evaluar(self, correo):
        texto = normalizar(f"{correo.asunto} {correo.cuerpo}")
        encontradas = [
            p for p in self.__palabras
            if re.search(rf"\b{re.escape(normalizar(p))}\w*", texto)
        ]
        if not encontradas:
            return 0, ""
        return len(encontradas) * self.peso, f"{self.nombre}: contiene {', '.join(encontradas)}"


class ReglaRemitente(Regla):
    """El dominio del remitente está en una lista negra (incluye subdominios)."""

    LISTA_NEGRA_POR_DEFECTO = ("promo-ya.com", "ganadores.net", "gratis.xyz")

    def __init__(self, lista_negra=None, peso=5):
        super().__init__("Remitente sospechoso", peso)
        self.__lista_negra = tuple(d.lower() for d in (lista_negra or self.LISTA_NEGRA_POR_DEFECTO))

    def evaluar(self, correo):
        dominio = correo.remitente.split("@")[-1].lower()
        for bloqueado in self.__lista_negra:
            # igual al dominio, o subdominio (mail.promo-ya.com); NO 'xpromo-ya.com'
            if dominio == bloqueado or dominio.endswith("." + bloqueado):
                return self.peso, f"{self.nombre}: remitente en lista negra ({bloqueado})"
        return 0, ""


class ReglaEnlaces(Regla):
    """El cuerpo del correo tiene demasiados enlaces."""

    def __init__(self, max_enlaces=2, peso=2):
        super().__init__("Enlaces", peso)
        self.__max_enlaces = max_enlaces

    def evaluar(self, correo):
        enlaces = re.findall(r"https?://\S+", correo.cuerpo, flags=re.IGNORECASE)
        if len(enlaces) > self.__max_enlaces:
            return self.peso, f"{self.nombre}: tiene {len(enlaces)} enlaces"
        return 0, ""


class ReglaMayusculas(Regla):
    """El asunto está en mayúsculas o tiene muchos signos de exclamación."""

    def __init__(self, peso=3):
        super().__init__("Mayúsculas y exclamaciones", peso)

    def evaluar(self, correo):
        asunto = correo.asunto
        letras = [c for c in asunto if c.isalpha()]
        motivos = []
        if len(letras) > 3 and asunto.isupper():
            motivos.append("asunto en mayúsculas")
        if asunto.count("!") >= 3:
            motivos.append("muchos signos de exclamación")
        if not motivos:
            return 0, ""
        return self.peso, f"{self.nombre}: {', '.join(motivos)}"


class ReglaNaiveBayes(Regla):
    """Regla opcional que APRENDE de ejemplos (Naive Bayes multinomial básico).

    Antes de entrenar (o si solo vio spam o solo normales) no opina: devuelve
    (0, ""). Una vez entrenada, si la probabilidad de spam del correo es
    mayor o igual que ``umbral_prob``, suma ``peso`` puntos como cualquier
    otra regla.
    """

    def __init__(self, peso=3, umbral_prob=0.5, alfa=1.0):
        super().__init__("Naive Bayes", peso)
        if not 0 < umbral_prob < 1:
            raise ValueError("umbral_prob debe estar entre 0 y 1")
        if alfa <= 0:
            raise ValueError("alfa debe ser mayor que 0")
        self.__umbral_prob = umbral_prob
        self.__alfa = alfa                           # suavizado de Laplace
        self.__n_correos = {True: 0, False: 0}       # correos vistos por clase
        self.__total_palabras = {True: 0, False: 0}  # palabras vistas por clase
        self.__frecuencias = {True: {}, False: {}}   # palabra -> veces, por clase
        self.__vocabulario = set()

    @property
    def entrenada(self):
        """True si ya vio al menos un correo spam y uno normal."""
        return self.__n_correos[True] > 0 and self.__n_correos[False] > 0

    def __palabras(self, correo):
        texto = normalizar(f"{correo.asunto} {correo.cuerpo}")
        return [p for p in re.findall(r"[a-z0-9]+", texto) if len(p) >= 3]

    def entrenar(self, ejemplos):
        """Aprende de una lista de pares (correo, es_spam). Se puede llamar varias veces."""
        for correo, es_spam in ejemplos:
            clase = bool(es_spam)
            self.__n_correos[clase] += 1
            for palabra in self.__palabras(correo):
                self.__frecuencias[clase][palabra] = self.__frecuencias[clase].get(palabra, 0) + 1
                self.__total_palabras[clase] += 1
                self.__vocabulario.add(palabra)

    def probabilidad_spam(self, correo):
        """Devuelve P(spam | correo) entre 0 y 1 (0.5 si aún no está entrenada)."""
        if not self.entrenada:
            return 0.5
        total_correos = self.__n_correos[True] + self.__n_correos[False]
        tamano_vocabulario = len(self.__vocabulario)
        log_prob = {}
        for clase in (True, False):
            lp = math.log(self.__n_correos[clase] / total_correos)
            denominador = self.__total_palabras[clase] + self.__alfa * tamano_vocabulario
            for palabra in self.__palabras(correo):
                if palabra in self.__vocabulario:
                    veces = self.__frecuencias[clase].get(palabra, 0)
                    lp += math.log((veces + self.__alfa) / denominador)
            log_prob[clase] = lp
        diferencia = max(-700, min(700, log_prob[False] - log_prob[True]))
        return 1 / (1 + math.exp(diferencia))

    def evaluar(self, correo):
        if not self.entrenada:
            return 0, ""
        prob = self.probabilidad_spam(correo)
        if prob >= self.__umbral_prob:
            return self.peso, f"{self.nombre}: probabilidad de spam {prob:.0%}"
        return 0, ""

**Resumen de `Regla` y sus hijas**

| Clase | Hereda de | Qué evalúa | Puntos que aporta |
|---|---|---|---|
| `Regla` | `ABC` | Clase base abstracta: guarda `nombre` y `peso` (debe ser mayor que 0) y obliga a las hijas a implementar `evaluar` | — |
| `ReglaPalabrasClave` | `Regla` | Palabras sospechosas en asunto y cuerpo (ignora mayúsculas y tildes; acepta plurales) | `peso` por cada palabra distinta (por defecto 2) |
| `ReglaRemitente` | `Regla` | Dominio del remitente en una lista negra (también sus subdominios) | `peso` (por defecto 5) |
| `ReglaEnlaces` | `Regla` | Más de `max_enlaces` enlaces en el cuerpo (por defecto 2) | `peso` (por defecto 2) |
| `ReglaMayusculas` | `Regla` | Asunto en mayúsculas o con 3 o más signos de exclamación | `peso` (por defecto 3) |
| `ReglaNaiveBayes` | `Regla` | Probabilidad de spam **aprendida de ejemplos** (Naive Bayes básico). Sin entrenar no opina | `peso` (por defecto 3) si la probabilidad es mayor o igual que `umbral_prob` (por defecto 0.5) |

**Métodos de `Regla` (los heredan todas las hijas)**

| Método | Qué hace |
|---|---|
| `__init__(nombre, peso)` | Guarda el nombre y el peso; valida que el peso sea positivo |
| `nombre`, `peso` | Propiedades de solo lectura |
| `evaluar(correo)` | **Abstracto.** Cada hija lo implementa y devuelve `(puntaje, motivo)`; si no se cumple, `(0, "")` |
| `__str__()` | Texto legible: nombre y peso |

**Métodos propios de `ReglaNaiveBayes`**

| Método | Qué hace |
|---|---|
| `entrenar(ejemplos)` | Aprende de una lista de pares `(correo, es_spam)`; se puede llamar varias veces |
| `entrenada` | `True` si ya vio al menos un correo spam y uno normal |
| `probabilidad_spam(correo)` | Devuelve P(spam \| correo) entre 0 y 1 (0.5 si aún no está entrenada) |

### 3.3 Clase `DetectorSpam`
Es el motor: guarda una lista de reglas, las aplica a un correo y decide si es spam. **No conoce los tipos concretos de reglas** (polimorfismo).

In [2]:
class DetectorSpam:
    """Contiene una lista de reglas y decide si un correo es spam."""

    def __init__(self, umbral=5):
        self.__reglas = []
        self.__umbral = umbral

    @property
    def umbral(self):
        return self.__umbral

    @umbral.setter
    def umbral(self, valor):
        if valor <= 0:
            raise ValueError("El umbral debe ser mayor que 0")
        self.__umbral = valor

    @property
    def reglas(self):
        return list(self.__reglas)

    def agregar_regla(self, regla):
        """Agrega una regla. Cualquier objeto que herede de Regla sirve."""
        if not isinstance(regla, Regla):
            raise TypeError("Solo se pueden agregar objetos Regla")
        self.__reglas.append(regla)

    def analizar(self, correo):
        """Evalúa el correo con todas las reglas y lo marca si supera el umbral."""
        correo.limpiar_motivos()
        total = 0
        for regla in self.__reglas:  # polimorfismo: no importa el tipo de regla
            puntaje, motivo = regla.evaluar(correo)
            total += puntaje
            if motivo:
                correo.agregar_motivo(motivo)

        correo.puntaje = total
        if total >= self.__umbral:
            correo.marcar_como_spam()
        else:
            correo.marcar_como_normal()
        return correo.es_spam

**Resumen de métodos de `DetectorSpam`**

| Método | Qué hace |
|---|---|
| `__init__(umbral)` | Crea el detector con una lista de reglas vacía y el umbral indicado |
| `umbral` (propiedad y setter) | Lee el umbral; al asignarlo, valida que sea mayor que 0 |
| `reglas` | Devuelve una copia de la lista de reglas |
| `agregar_regla(regla)` | Agrega una regla; valida que sea un objeto `Regla` (o de una clase hija) |
| `analizar(correo)` | Aplica todas las reglas, suma los puntajes, guarda puntaje y motivos en el correo y lo marca spam o normal según el umbral |

### 3.4 Clase `Bandeja`
Almacena los correos y responde consultas. **No analiza** (eso es del detector) ni imprime (eso es del menú).

## 4. Ejecución de prueba

Se usan **todas** las clases en secuencia.

**Paso 1: crear el detector y registrar las reglas**

**Paso 2: crear la bandeja y cargar correos de ejemplo** (spam claros, normales y casos dudosos)

**Paso 3: analizar todos los correos**

**Paso 4: mostrar los resultados** (spam con sus motivos y correos normales)

**Paso 5: demostrar herencia, polimorfismo y encapsulamiento**


*Encapsulamiento y validaciones:* intentar romper las reglas de la clase produce errores controlados.

## 5. Menú interactivo 

Menú en consola con `input()`.

## 6. Informe: clases, atributos y métodos


## 7. Conceptos de POO aplicados


## 8. Conclusiones